In [1]:
import pandas as pd
import random
import csv
from tqdm import tqdm
import string

from collections import Counter

import spacy
# spacy.cli.download("nl_core_news_lg")
nlp = spacy.load("nl_core_news_lg") 

import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ['hf_token_own']
)

In [2]:
path_name = '/Users/sabijn/Documents/PhD/Datasets/chisor_dataset_all/ChiSCor_CoNLL_paper/csv/ChiSCor_master_df_password/ChiSCor_master_df.csv'
df = pd.read_csv(path_name, index_col=0)

In [3]:
pos_tags = []
pos_tag_story_begin = []
for story in df['story_raw']:
    doc = nlp(story)

    sent_list = list(doc.sents)

    for i, sent in enumerate(sent_list):
        first_word = sent[0]
        if i == 0:
            pos_tag_story_begin.append(first_word.pos_)

        pos_tags.append(first_word.pos_)
pos_counter = Counter(pos_tag_story_begin)

In [4]:
nouns = set()
adjectives = set()
verbs = set()

for story in df['story_raw']:
    for token in nlp(story):
        if token.pos_ == 'NOUN':
            nouns.add(str(token))
        elif token.pos_ == 'VERB':
            verbs.add(str(token))
        elif token.pos_ == 'ADJ':
            adjectives.add(str(token))

In [5]:
nouns, adjectives, verbs = list(nouns), list(adjectives), list(verbs)

In [6]:
def select_pos_tag(weighted=True):
    if weighted:
        tags = list(pos_counter.keys())
        frequencies = list(pos_counter.values())
        sample = random.choices(tags, weights=frequencies, k=1)[0]
    else:
        sample = pos_tags[random.randint(0, len(pos_tags) - 1)]
    
    return sample

In [7]:
# you took these from Finke (2025), translated them, and removed a few. 
story_features = [
    "dialoog",
    "in medias res",
    "een morele les",
    "een onverwachte wending",
    "een onbetrouwbare verteller",
    "vooruitwijzing",
    "ironie",
    "innerlijke monoloog",
    "symboliek",
    "een MacGuffin",
    "een niet-lineaire tijdlijn",
    "een omgekeerde tijdlijn",
    "circulaire verhaalsstructuur",
    "een flashback",
    "een geneste structuur",
    "een dwaalspoor",
    "meerdere perspectieven",
    "de vierde wand",
    "een cliffhanger",
    "een antiheld",
    "contrast (juxtapositie)",
    "climax-structuur"
]

verhaalelementen = [
    "sprekende dieren",
    "fantasiewerelden",
    "tijdreizen",
    "een deadline of tijdslimiet",
    "ruimteverkenning",
    "mystieke wezens",
    "onderwateravonturen",
    "dinosaurussen",
    "piraten",
    "superhelden",
    "sprookjes",
    "het heelal",
    "verborgen schatten",
    "magische landen",
    "betoverde bossen",
    "geheime genootschappen",
    "robots en technologie",
    "sport",
    "schoolleven",
    "vakanties",
    "culturele tradities",
    "magische voorwerpen",
    "verloren beschavingen",
    "ondergrondse werelden",
    "vervlogen tijdperken",
    "onzichtbaarheid",
    "reusachtige wezens",
    "miniatuurwerelden",
    "ontmoetingen met buitenaardse wezens",
    "behekste plekken",
    "vormverandering",
    "eilandavonturen",
    "ongewone voertuigen",
    "geheime missies",
    "droomwerelden",
    "virtuele werelden",
    "raadsels",
    "rivaliteit tussen broers en zussen",
    "schattenjachten",
    "sneeuwavonturen",
    "seizoenswisselingen",
    "mysterieuze kaarten",
    "koninkrijken",
    "levende objecten",
    "tuinen",
    "verloren steden",
    "de kunsten",
    "de hemel"
]

In [71]:
# Rags to Riches Riches to Rags Man in a Hole Double Man in a Hole Icarus Cinderella Oedipus (macro niveau)

# TP1 - Opportunity TP2 - Change of Plans TP3 - Point of No Return TP4 - Major Setback TP5 - Climax The introductory event that sets the stage for the narrative.
# A pivotal moment where the main goal of the narrative is defined or altered.
# The commitment point beyond which the protagonists are invested in goals.
# A critical juncture where the protagonists face significant challenges or failures.
# The peak of the narrative arc, encompassing the resolution of the central conflict.

In [ ]:
def generate_user_prompt():
    chosen_noun = random.choice(nouns)
    chosen_adjective = random.choice(adjectives)
    chosen_verb = random.choice(verbs)
    chosen_pos_tag = select_pos_tag()
    chosen_letter = random.choice(string.ascii_lowercase)
    chosen_feature = random.choice(story_features)
    element = random.choice(verhaalelementen)

    prompt = f"""Vertel een verhaal. 
    Het verhaal moet het volgende werkwoord bevatten: {chosen_verb}, het volgende zelfstandig naamwoord: {chosen_noun} en het volgende bijvoegelijk naamwoord: {chosen_adjective}.
    Het verhaal moet het volgende kenmerk bevatten: {chosen_feature} en het volgende verhaal element: {element}.
    Begin het verhaal met een woord met het volgende pos-tag {chosen_pos_tag}."""

    return prompt

In [ ]:
prompt = generate_user_prompt()

In [ ]:
lower_age = 4
upper_age = 6
system_prompt = f"""
                Je bent een verteller van een kort verhaal (rond de 100 woorden).
                Je bent een kind tussen de {lower_age} en {upper_age} en je vertelt een verhaal aan klasgenoten. Gebruik woorden en taalconstructies die kinderen van die leeftijd gebruiken. 
                Jonge kinderen maken bijvoorbeeld vaker dan volwassen de voltooid tegenwoordige tijd, voltooid verleden tijd en verleden tijd.
                Geef het verhaal geen titel of introductie. Het verhaal hoeft geen ego-narratie te zijn, mensen gebruiken een verhaal zelden om hun eigen perspectief te vertellen. Het mag dus verteld worden
                vanuit het perspectief van iemand anders.
                """

In [ ]:
with open("/Users/sabijn/Documents/PhD/code/storylm_p1_data/prompting/....csv", "a", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header only once if file is empty
    csvfile.seek(0, 2)  # move to end
    if csvfile.tell() == 0:  
        writer.writerow(["model", "prompt", "completion"])

    for _ in tqdm(range(5)):
        print(prompt)
        response = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct:novita",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt},
            ],
        )

        completion = response.choices[0].message.content.strip()
        print(completion)

        # Write each row immediately
        writer.writerow([response.model, prompt, completion])


  0%|          | 0/5 [00:00<?, ?it/s]

prompt: Vertel een verhaal. 
    Het verhaal moet het volgende werkwoord bevatten: doet, het volgende zelfstandig naamwoord: gevechtsruimteschepen en het volgende bijvoegelijk naamwoord: wiee.
    Het verhaal moet het volgende kenmerk bevatten: een morele les en het volgende verhaal element: verborgen schatten.
    Begin het verhaal met een woord met het volgende pos-tag ADV.


 20%|██        | 1/5 [00:00<00:03,  1.33it/s]

Snel liep Sam met zijn vriendjes door de zandbak. Ze speelden piraten, want piraten hebben altijd verborgen schatten. “Kijk hier!” riep Sam. Hij had een kaart gevonden in een oude doos. Op de kaart stond een X bij een grote baksteen.  

De jongens gingen erheen. Terwijl ze de baksteen omwoelden, zei Lotte: “Wat is dat raar geluid?” Een wiee‑geluid kwam uit de grond. Het klonk als een zacht “wiee‑wiee”.  

Plotseling kwam er een glimmend, miniatuur gevechtsruimteschepen uit de zandkorrel tevoorschijn. Het schip deed een pirouette en spoot een straal van glinsterend zand. “Wow!” schreeuwde Tim.  

In het schip zat een klein, gouden muntje. Maar er waren nog meer munten verstopt in een holte onder het zand. De jongens vonden een hele berg verborgen schatten! Ze sprongen blij rond en deelden de munten.  

Toen kwam de lerares naar buiten. “Kinderen, wat doen jullie met al die schatten?” vroeg ze. Sam vertelde dat ze ze hadden gevonden. De lerares glimlachte. “Het is fijn om te vinden, maar

 40%|████      | 2/5 [00:02<00:03,  1.09s/it]

...

Ik vertelde het verhaal aan de klas.  

Op een dag ging ik met de juf naar een boerderij. We zagen een grote **stal** met **verschillende** dieren. In de stal stonden drie paarden. Eén paard had een rode halsband en de andere twee hadden blauwe. De juf zei: “Kijk, de paarden zijn als een sportteam.”  

We gingen daarna naar het veld. Ik **plukte** een bosje wilde bloemen bij de rand en gaf het aan mijn vriend Tom. Hij lachte en zei dat de bloemen als een trofee waren voor ons spel.  

Daarna speelden we voetbal. De bal rolde over het gras en we renden als de paarden in de stal. Het roze paard rende snel, het blauwe paard bleef rustig, en het rode paard sprong hoog. Het voelde alsof elk paard een verschillende manier van spelen liet zien.  

Na een tijdje viel het regentje. De juf zei dat de regen een symbool was voor samenwerking. “Als de regen ons nat maakt, dan moeten we elkaar helpen, net als de paarden die samen in de stal staan.” We gingen onder de stal zitten, want de stal was ee

 80%|████████  | 4/5 [00:04<00:01,  1.15s/it]

Ik vertelde jullie iets heel geks dat ik gisteren heb meegemaakt.  

We gingen met klasgenoot Tom en zijn **hondje** naar het bos, want Tom had gezegd dat daar een oude boom stond waar een schat onder begraven zou zitten. Het **hondje** sprong blij rond en blafte hard, en ik dacht dat het de **allersterkste** speurneus had.  

Toen we bij de boom kwamen, vond ik een oud, verroeste kaart die in een gat lag. De kaart liet zien dat er drie **verborgen schatten** onder de boom lagen, maar alleen als je vier stappen naar links ging, daarna twee stappen terug, en dan **als je** heel stil staat en naar het geluid van de wind luistert.  

Ik zei tegen Tom: “**Vertel** je vriendjes dat we nu zoeken, maar eerst moeten we weten hoe we de schat vinden.” Tom knikte en het **hondje** begon te graven. Terwijl het hondje graaft, hoorde ik een zacht geruis van de wind die door de bladeren fluisterde.  

In dat moment, **waar** ik dacht dat we niets zouden vinden, verscheen een klein, glinsterend kistje

100%|██████████| 5/5 [00:05<00:00,  1.16s/it]

Ik zat op het schoolplein en keek naar de blauwe lucht. Ik dacht aan de vakantie van gister, toen we met de hele familie naar het strand gingen. Het was zo fijn, want we maakten zandkastelen en speelden verstoppertje tussen de golven.

Even toen ik terug dacht, kreeg ik een flashback. Ik zag mezelf als klein kind, heel klein, die op de rand van het water stond. De rand voelde krap en ik hield mijn hand strak om papa’s veters. Ik herinnerde me hoe ik bijna in het water viel, maar toen kwam een dikke dolfijn, en die trok me zachtjes terug. Ik had heel hard gelachen.

Na die flashback ging ik weer naar de klas. De juf vroeg: “Wie heeft iets leuks verzon tijdens de vakantie?” Ik zei dat ik een piratenboot verzon die op de rand van een eiland rustte. We knutselden die boot van karton en gaf hem een vlag met een grote ster. Iedereen vond het leuk.

De vakantie van gister was ook het einde van het schooljaar. We hadden een grote picknick en speelden tot het donker werd. Ik vertelde mijn vrien

In [ ]:
# Je bent een verteller van een kort verhaal (ongeveer 200 woorden).
#                 Je bent een kind tussen de 4 en 6 jaar oud en je vertelt een verhaal aan klasgenoten. 
#                 Je publiek bestaat ook uit kinderen van jouw leeftijd.
#                 Geef het verhaal geen titel of introductie. 

# iets over eenvoudig lexicon
# iets over onvoltooid verleden tijd
# iets over vertelperspectief
